# HTTP with requests: GET, POST, Status Codes, Clients

## What you'll practice

Every function takes a `session` argument instead of calling `requests.get` directly. In the notebook you pass `MockSession()` (offline) or a real
`requests.Session()`; in tests a `MockSession` with injected faults. That's dependency injection — Python style: no framework, just a parameter.
Always pass `timeout=`; `requests` waits forever by default.

> Function and class names in **Implement** are what the tests look for. Run all tests for this notebook with `pytest tests/apis/test_07_01_http_requests_basics.py -v` from the lab root. Untouched exercises show as *skipped*.
>
> Solutions: `solutions/07_apis/01_http_requests_basics.ipynb`. Don't open it until you've tried.

### Setup — run this first (sample data and imports, not a solution)

In [ ]:
import sys
from pathlib import Path
LAB = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets").is_dir() and (p / "labkit").is_dir())   # lab root (works from any folder)
sys.path.insert(0, str(LAB))                                # make `labkit` importable from the notebook
from labkit.mock_api import MockSession, BASE_URL
import requests

session = MockSession()                                     # offline fake with the same surface as requests.Session
resp = session.get(f"{BASE_URL}/customers", params={"page": 1, "page_size": 3}, timeout=5)
print(resp.status_code, resp.json()["total"], [c["customer_id"] for c in resp.json()["data"]])
# Prefer real HTTP? In a terminal:  python -m labkit.mock_server --port 8765
# then use requests.Session() with base_url "http://127.0.0.1:8765/v1" — your functions shouldn't change at all.

## Exercise 1 — 🟢 Easy · GET a resource, 404 → None

**Implement:** `get_customer(session, base_url: str, customer_id: str) -> dict | None`

GET `{base_url}/customers/{customer_id}` with `timeout=10`. Return the JSON dict on 200, `None` on 404, and call `raise_for_status()` for any other error status.

**Input:** `session, base_url: str, customer_id: str`

**Expected output:** `dict | None`

**Constraints:** Standard library + `requests`-compatible session objects; always pass `timeout=`.

**Example:** the tests are executable examples — read `test_get_customer, test_get_customer_server_error` in `tests/apis/test_07_01_http_requests_basics.py`.

**Check:** `pytest tests/apis/test_07_01_http_requests_basics.py -k "test_get_customer or test_get_customer_server_error"`

**YOUR CODE ↓**

## Exercise 2 — 🟡 Medium · POST JSON and handle validation errors

**Implement:** `create_order(session, base_url: str, customer_id: str, items: list[dict]) -> int`

POST `{"customer_id", "items"}` as JSON to `/orders`. Return the new `order_id` on 201. On 422 raise `ValueError` whose message joins the API's `errors` list with `"; "`. Other failures → `raise_for_status()`.

**Input:** `session, base_url: str, customer_id: str, items: list[dict]`

**Expected output:** `int`

**Constraints:** Standard library + `requests`-compatible session objects; always pass `timeout=`.

**Example:** the tests are executable examples — read `test_create_order, test_create_order_validation` in `tests/apis/test_07_01_http_requests_basics.py`.

**Check:** `pytest tests/apis/test_07_01_http_requests_basics.py -k "test_create_order or test_create_order_validation"`

**YOUR CODE ↓**

## Exercise 3 — 🟡 Medium · Build URLs with query parameters safely

**Implement:** `build_url(base: str, path: str, **params) -> str`

Join `base` and `path` with exactly one `/`. Drop params whose value is `None`; list/tuple values become repeated params (`status=A&status=B`);
booleans become `true`/`false`. Encode with `urllib.parse.urlencode(..., doseq=True)`; keep keyword order. No `?` when there are no params.

**Input:** `base: str, path: str, **params`

**Expected output:** `str`

**Constraints:** Standard library + `requests`-compatible session objects; always pass `timeout=`.

**Example**

```python
build_url("https://api.x/v1/", "/orders", status=["NEW", "PAID"], limit=50, cursor=None, active=True)
# 'https://api.x/v1/orders?status=NEW&status=PAID&limit=50&active=true'
```

**Check:** `pytest tests/apis/test_07_01_http_requests_basics.py -k test_build_url`

**YOUR CODE ↓**

## Exercise 4 — 🟠 Hard · A typed API client with an exception hierarchy

**Implement:** `class ApiError(Exception)`, `class ApiClient`

`ApiClient(session, base_url, api_key=None, timeout=5)`; `get_json(path, **params)` and `post_json(path, body)`. Send `Authorization: Bearer <key>` when a key is set
and `Accept: application/json`. Map failures to exceptions, all subclasses of `ApiError` (which has `.status` and `.payload`):
`AuthError` (401/403), `NotFound` (404), `RateLimited` (429, with `.retry_after` float from the header or `None`), `ServerError` (5xx), plain `ApiError` for other 4xx.
Network timeouts propagate as `requests.Timeout` (don't wrap them).

**Input:** see the class description above

**Expected output:** an instance behaving as described

**Constraints:** Standard library + `requests`-compatible session objects; always pass `timeout=`.

**Example:** the tests are executable examples — read `test_api_client_auth_and_success, test_api_client_error_mapping` in `tests/apis/test_07_01_http_requests_basics.py`.

**Check:** `pytest tests/apis/test_07_01_http_requests_basics.py -k "test_api_client_auth_and_success or test_api_client_error_mapping"`

**YOUR CODE ↓**

## Challenge — 🔴 Very Hard · Concurrent lookups with a thread pool

**Implement:** `fetch_many(session, base_url: str, ids: list[str], max_workers: int = 8) -> tuple[dict, list[str], dict]`

Fetch many customers concurrently (I/O-bound → threads help). Use `concurrent.futures.ThreadPoolExecutor`. Return `(found, missing, failed)`:
`found` = `{id: record}`, `missing` = sorted ids that returned 404, `failed` = `{id: "ExceptionType: message"}` for anything else (timeouts, 5xx).
Duplicate ids are fetched once. One failing request must not cancel the rest.

**Input:** `session, base_url: str, ids: list[str], max_workers: int = 8`

**Expected output:** `tuple[dict, list[str], dict]`

**Constraints:** Standard library + `requests`-compatible session objects; always pass `timeout=`.

**Example:** the tests are executable examples — read `test_fetch_many` in `tests/apis/test_07_01_http_requests_basics.py`.

**Check:** `pytest tests/apis/test_07_01_http_requests_basics.py -k test_fetch_many`

**YOUR CODE ↓**